# Stream A — QC + Feature Analysis

Three tasks:
1. **Waveform QC** — raw audio comparison, normal vs depressed (PHQ-8), annotate outliers
2. **F0 + Energy QC plot** — pitch variability alongside energy features across all subjects
3. **Z-score normalization** — standardize all features before classification

---
## Cell 0 — Configuration

In [ ]:
from pathlib import Path
RAW_DATA_DIR  = Path('/Users/Meihui/Downloads/sync/work/whisperx')  # ✏️
FEATURES_CSV  = Path('outputs/features/improved_features.csv')
LABELS_DIR    = RAW_DATA_DIR
QC_OUTPUT_DIR = Path('outputs/qc')
QC_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PHQ8_THRESHOLD = 10
TARGET_SR      = 16000
WAVEFORM_SECS  = 60
print(f'Features CSV: {FEATURES_CSV}')
print(f'QC output:    {QC_OUTPUT_DIR.resolve()}')

---
## Cell 1 — Load Features + PHQ-8 Labels

In [ ]:
import numpy as np
import pandas as pd
import warnings; warnings.filterwarnings('ignore')

df = pd.read_csv(FEATURES_CSV)
print(f'Features loaded: {df.shape[0]} subjects x {df.shape[1]} columns')

if 'PHQ8_Score' not in df.columns:
    dfs = []
    for fname in ['train_split_Depression_AVEC2017.csv', 'dev_split_Depression_AVEC2017.csv']:
        fpath = LABELS_DIR / fname
        if fpath.exists():
            tmp = pd.read_csv(fpath)
            tmp['subject_id'] = tmp['Participant_ID'].astype(str) + '_P'
            dfs.append(tmp)
    if dfs:
        labels = pd.concat(dfs)[['subject_id','PHQ8_Binary','PHQ8_Score','Gender']]
        df = df.merge(labels, on='subject_id', how='left')
    else:
        df['PHQ8_Score'] = np.nan; df['PHQ8_Binary'] = np.nan; df['Gender'] = np.nan

df['group'] = df['PHQ8_Score'].apply(
    lambda x: 'depressed' if pd.notna(x) and x >= PHQ8_THRESHOLD else ('normal' if pd.notna(x) else 'unknown'))
print(df[['subject_id','PHQ8_Score','group']].to_string(index=False))

---
## Cell 2 — Task 1: Waveform QC
Compare raw audio of depressed vs normal subjects. Annotate amplitude outliers.

In [ ]:
import librosa
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

def find_audio(subject_id):
    d = RAW_DATA_DIR / subject_id
    audio = d / f"{subject_id.replace('_P','')}_AUDIO.wav"
    if not audio.exists():
        wavs = sorted(d.glob('*.wav'))
        if wavs: return wavs[0]
    return audio

waveform_data = {}
for _, row in df.iterrows():
    sid = row['subject_id']
    path = find_audio(sid)
    if not path.exists(): continue
    audio, _ = librosa.load(str(path), sr=TARGET_SR, mono=True, duration=WAVEFORM_SECS)
    rms = librosa.feature.rms(y=audio)[0]
    waveform_data[sid] = {'audio': audio, 'peak': float(np.max(np.abs(audio))),
                          'rms_mean': float(np.mean(rms)), 'group': row['group'], 'phq8': row['PHQ8_Score']}

rms_vals = np.array([v['rms_mean'] for v in waveform_data.values()])
rms_m, rms_s = np.mean(rms_vals), np.std(rms_vals)
for sid, s in waveform_data.items():
    s['amplitude_outlier'] = abs(s['rms_mean'] - rms_m) > 2 * rms_s

normal_df = df[df['group']=='normal'].sort_values('PHQ8_Score')
depressed_df = df[df['group']=='depressed'].sort_values('PHQ8_Score', ascending=False)
ref_sid  = normal_df.iloc[0]['subject_id'] if len(normal_df) > 0 else df.iloc[0]['subject_id']
comp_sid = depressed_df.iloc[0]['subject_id'] if len(depressed_df) > 0 else df.iloc[1]['subject_id']
print(f'Reference (normal)    : {ref_sid}  PHQ-8={df[df.subject_id==ref_sid].PHQ8_Score.values[0]}')
print(f'Comparison (depressed): {comp_sid}  PHQ-8={df[df.subject_id==comp_sid].PHQ8_Score.values[0]}')

fig, axes = plt.subplots(3, 2, figsize=(16, 11))
for col, (sid, color) in enumerate([(ref_sid,'steelblue'), (comp_sid,'tomato')]):
    s = waveform_data.get(sid); audio = s['audio']
    t = np.linspace(0, len(audio)/TARGET_SR, len(audio))
    outlier_note = ' ⚠️ AMPLITUDE OUTLIER' if s['amplitude_outlier'] else ''
    axes[0,col].plot(t, audio, color=color, linewidth=0.3, alpha=0.8)
    axes[0,col].set_ylim(-1.05, 1.05)
    axes[0,col].set_title(f"{sid}  PHQ-8={s['phq8']}  {s['group'].upper()}{outlier_note}", fontsize=10, fontweight='bold', color=color)
    axes[0,col].text(0.02, 0.92, f"Peak={s['peak']:.3f}  RMS={s['rms_mean']:.4f}", transform=axes[0,col].transAxes, fontsize=9, bbox=dict(boxstyle='round',facecolor='white',alpha=0.85))
    rms_f = librosa.feature.rms(y=audio, frame_length=1024, hop_length=512)[0]
    rt = librosa.frames_to_time(np.arange(len(rms_f)), sr=TARGET_SR, hop_length=512)
    axes[1,col].plot(rt, rms_f, color=color, linewidth=0.8)
    axes[1,col].axhline(np.mean(rms_f), color='black', linestyle='--', linewidth=1, label=f'mean={np.mean(rms_f):.4f}')
    axes[1,col].set_title(f'{sid} — RMS Energy over Time', fontsize=10); axes[1,col].legend(fontsize=8)
    axes[2,col].hist(audio, bins=100, color=color, alpha=0.7, edgecolor='none')
    axes[2,col].set_title(f'{sid} — Amplitude Distribution', fontsize=10)
plt.tight_layout()
plt.savefig(QC_OUTPUT_DIR / 'qc_waveform_phq8.png', dpi=150, bbox_inches='tight'); plt.show()
print('\nAmplitude outlier summary:')
for sid, s in waveform_data.items():
    flag = ' <-- OUTLIER ⚠️' if s['amplitude_outlier'] else ''
    print(f"  {sid:<12} PHQ-8={s['phq8']:>5}  RMS={s['rms_mean']:.4f}  {s['group']:<10}{flag}")

---
## Cell 3 — Task 2: F0 + Energy QC Plot
Pitch variability alongside energy. Bars coloured by PHQ-8 group. PHQ-8 score annotated below each bar.

In [ ]:
subjects = df['subject_id'].tolist()
groups   = df['group'].tolist()
phq8s    = df['PHQ8_Score'].tolist()
x = np.arange(len(subjects))
GROUP_COLORS = {'normal': 'steelblue', 'depressed': 'tomato', 'unknown': 'grey'}
bar_colors = [GROUP_COLORS[g] for g in groups]

FEATURES = [
    # F0 — pitch (most directly linked to anhedonia)
    ('F0semitoneFrom27.5Hz_sma3nz_amean_mean',      'F0 Mean (semitone)',                        'F0'),
    ('F0semitoneFrom27.5Hz_sma3nz_stddevNorm_mean', 'F0 Std Norm  ← PITCH VARIABILITY',          'F0'),
    ('F0semitoneFrom27.5Hz_sma3nz_percentile20.0_mean', 'F0 20th Percentile',                    'F0'),
    ('F0semitoneFrom27.5Hz_sma3nz_percentile80.0_mean', 'F0 80th Percentile',                    'F0'),
    # Energy
    ('loudness_sma3_amean_mean',      'Loudness Mean',      'Energy'),
    ('loudness_sma3_stddevNorm_mean', 'Loudness Std Norm',  'Energy'),
    # Voice quality
    ('HNRdBACF_sma3nz_amean_mean',     'HNR — voice quality',       'Quality'),
    ('jitterLocal_sma3nz_amean_mean',  'Jitter — voice irregularity', 'Quality'),
]
available = [(col, label, grp) for col, label, grp in FEATURES if col in df.columns]
GROUP_SECTION_COLORS = {'F0': '#FF8C00', 'Energy': '#2196F3', 'Quality': '#9C27B0'}

fig, axes = plt.subplots(len(available), 1, figsize=(15, len(available) * 3.5))
if len(available) == 1: axes = [axes]
plt.subplots_adjust(hspace=0.6)

for i, (col, label, grp) in enumerate(available):
    vals = df[col].values.astype(float)
    gm, gs = np.nanmean(vals), np.nanstd(vals)
    is_outlier = np.abs(vals - gm) > 2 * gs
    colors_i = ['orange' if is_outlier[j] else bar_colors[j] for j in range(len(subjects))]
    axes[i].bar(x, vals, color=colors_i, alpha=0.85, edgecolor='white', linewidth=0.5)
    for j, (val, outlier) in enumerate(zip(vals, is_outlier)):
        if outlier:
            offset = 0.03 * (np.nanmax(vals) - np.nanmin(vals))
            axes[i].text(j, val + offset, '⚠️', ha='center', va='bottom', fontsize=9)
    for j, (sid, phq) in enumerate(zip(subjects, phq8s)):
        label_txt = f'{phq:.0f}' if pd.notna(phq) else '?'
        axes[i].annotate(label_txt, xy=(j, 0), xycoords=('data', 'axes fraction'),
                         xytext=(0, -22), textcoords='offset points',
                         ha='center', va='top', fontsize=6.5, color='dimgrey')
    axes[i].axhline(gm, color='black', linestyle='--', linewidth=1.0, label=f'mean={gm:.4f}')
    axes[i].axhspan(gm-gs, gm+gs, alpha=0.07, color='black', label='±1 SD')
    axes[i].set_title(f"[{grp}]  {label}", fontsize=9, fontweight='bold', color=GROUP_SECTION_COLORS.get(grp,'black'), loc='left')
    axes[i].set_xticks(x); axes[i].set_xticklabels(subjects, rotation=35, ha='right', fontsize=8)
    axes[i].legend(fontsize=7, loc='upper right')

legend_patches = [mpatches.Patch(color='steelblue', label='Normal (PHQ-8 < 10)'),
                  mpatches.Patch(color='tomato',    label='Depressed (PHQ-8 ≥ 10)'),
                  mpatches.Patch(color='orange',    label='Outlier (>2 SD)'),
                  mpatches.Patch(color='grey',      label='Unknown')]
fig.legend(handles=legend_patches, loc='upper right', bbox_to_anchor=(1.15, 1.0), fontsize=9, framealpha=0.9)
plt.suptitle('QC: F0 (Pitch) + Energy Features\nBar color = PHQ-8 group  |  ⚠️ = >2 SD  |  Number below bar = PHQ-8 score',
             fontsize=12, fontweight='bold', y=1.02)
plt.savefig(QC_OUTPUT_DIR / 'qc_f0_energy_phq8.png', dpi=150, bbox_inches='tight'); plt.show()

---
## Cell 4 — Task 3: Z-score Normalization
Standardize all features before classification. RMS (~0.001–0.07) and MFCC (~-500 to -100) are on completely different scales.

In [ ]:
from sklearn.preprocessing import StandardScaler

META_COLS = ['subject_id','PHQ8_Binary','PHQ8_Score','Gender','split','group','n_segments','total_speech_sec']
meta_cols_present = [c for c in META_COLS if c in df.columns]
feature_cols = [c for c in df.columns if c not in meta_cols_present]
print(f'Feature columns to normalize: {len(feature_cols)}')

scaler   = StandardScaler()
X_scaled = scaler.fit_transform(df[feature_cols].values.astype(float))
df_scaled = df[meta_cols_present].copy()
df_scaled[feature_cols] = X_scaled

means = df_scaled[feature_cols].mean()
print(f'Verification — max absolute mean: {means.abs().max():.6f}  (should be ~0)')
print(f'              max absolute std : {df_scaled[feature_cols].std().max():.4f}  (should be ~1)')

---
## Cell 5 — Save Normalized CSV + Diagnostic Plots

In [ ]:
out_csv = Path('outputs/features/improved_features_zscored.csv')
df_scaled.to_csv(out_csv, index=False)
print(f'Saved: {out_csv}  |  Shape: {df_scaled.shape}')

sample_cols = [c for c in feature_cols if any(k in c for k in ['loudness','F0semitone','jitter','HNR'])][:6]
short_names = [c.replace('F0semitoneFrom27.5Hz_sma3nz_','F0_').replace('loudness_sma3_','loud_').replace('_mean','') for c in sample_cols]
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].boxplot([df[c].dropna().values for c in sample_cols], labels=short_names, patch_artist=True)
axes[0].set_title('Before Z-score', fontsize=11, fontweight='bold'); axes[0].tick_params(axis='x', rotation=30)
axes[1].boxplot([df_scaled[c].dropna().values for c in sample_cols], labels=short_names, patch_artist=True)
axes[1].set_title('After Z-score', fontsize=11, fontweight='bold')
axes[1].axhline(0, color='black', linestyle='--', linewidth=0.8); axes[1].tick_params(axis='x', rotation=30)
plt.suptitle('Feature Scale: Before vs After Z-score Normalization', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(QC_OUTPUT_DIR / 'qc_zscore_comparison.png', dpi=150, bbox_inches='tight'); plt.show()
print('\nReady for classification. Use: improved_features_zscored.csv')